In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, IntegerType, DateType


qui si pescano i nuovi dati da source (source si sovrascrive) e si mandano in delta sul bronze

## acheck adls folders

In [0]:
dbutils.fs.ls("abfss://source@storageaccountmeteo.dfs.core.windows.net/")

In [0]:
dbutils.fs.ls("abfss://source@storageaccountmeteo.dfs.core.windows.net/tables/")


### estrai tabelle da adls

In [0]:
# 1. Get the list of all files/folders in the path
raw_files = dbutils.fs.ls("abfss://source@storageaccountmeteo.dfs.core.windows.net/tables/")

# 2. Extract only the names
# We use .rstrip('/') because folder names in ADLS often end with a slash
tabelle = [file.name.rstrip('/') for file in raw_files]

# 3. Filter the list (if you only want specific tables)
# For example, if you want to skip the metadata table or hidden files
tabelle_clean = [t for t in tabelle if "weather" in t and not t.startswith("_")]

print(f"Tables found: {tabelle_clean}")

## autoloader

In [0]:
source_base_path = "abfss://source@storageaccountmeteo.dfs.core.windows.net/tables"

for t in tabelle_clean:
    
    s_path = f"{source_base_path}/{t}/"
    
    # Ingestione BATCH
    (spark.read
        .format("csv")
        .option("header", "true")
        .option("inferSchema", "true") # Sostituisce inferColumnTypes dello streaming
        .load(s_path)
        .write
        .format("delta")
        .mode("overwrite") # Sovrascrive la tabella Bronze con i file attuali su ADLS
        .option("overwriteSchema", "true")
        .saveAsTable(f"catalogmeteo.bronze.{t}")
    )
